In [ ]:
# --- PART E: VALUE LOOP ANALYSIS (UPDATED) ---

# Method: Computational Brute-Force (Cycle Detection)
# We use NetworkX to identify all elementary circuits (simple cycles) in the graph.
# A "Loop Score" is calculated by summing the weights (Importance * Benefit) of all edges in the cycle.

import networkx as nx
import pandas as pd

# 1. Initialize the Directed Graph
G = nx.DiGraph()

# 2. Define all Stakeholders (Nodes)
# Updated to include separated Pilots/Owners and new Passengers
nodes = [
    "System", "Pilots", "Owners", "Pax", "Schools", "Mfg", "FAA",
    "Govt", "Insurers", "MRO", "Airports", "Investors", "Comm"
]

# 3. Define Edges with Scores (Source, Target, Weight)
# Weight = Importance x Benefit (from your updated Table D)
edges = [
    # --- Radial Flows (System <-> Stakeholder) ---
    # 1. Pilots (Feedback Loop)
    ("System", "Pilots", 9), ("Pilots", "System", 4), # Pilots give feedback (score 4), not revenue

    # 2. Owners (Revenue Loop)
    ("System", "Owners", 9), ("Owners", "System", 9), # Owners pay purchase price (score 9)

    # 3. Flight Schools
    ("System", "Schools", 9), ("Schools", "System", 9),

    # 4. Passengers (Pax)
    ("System", "Pax", 9), ("Pax", "System", 9),

    # 5. Manufacturing Partner
    ("System", "Mfg", 6), ("Mfg", "System", 9),

    # 6. Regulators (FAA)
    ("System", "FAA", 6), ("FAA", "System", 9),

    # 7. Govt Sponsor
    ("System", "Govt", 4), ("Govt", "System", 6),

    # 8. Insurers
    ("System", "Insurers", 4), ("Insurers", "System", 4),

    # 9. MRO (Maintenance)
    ("System", "MRO", 6), ("MRO", "System", 4),

    # 10. Airports
    ("System", "Airports", 2), ("Airports", "System", 4),

    # 11. Investors
    ("System", "Investors", 9), ("Investors", "System", 9),

    # 12. Community
    ("System", "Comm", 2), ("Comm", "System", 1),

    # --- Rim Flows (Direct Exchanges Between Stakeholders) ---
    # Owners pay Insurers
    ("Owners", "Insurers", 6), ("Insurers", "Owners", 9),

    # Owners pay Airports
    ("Owners", "Airports", 2), ("Airports", "Owners", 9),

    # Schools pay MRO
    ("Schools", "MRO", 6), ("MRO", "Schools", 9),

    # Investors fund Manufacturer
    ("Investors", "Mfg", 9), ("Mfg", "Investors", 6),

    # Passengers pay Owners (Fares)
    ("Pax", "Owners", 9), ("Owners", "Pax", 9)
]

# Add edges to the graph
G.add_weighted_edges_from(edges)

# 4. Find all Simple Cycles (Loops)
# simple_cycles finds all elementary circuits (loops without repeated nodes)
raw_cycles = list(nx.simple_cycles(G))

# 5. Filter and Score the Loops
loop_data = []

for cycle in raw_cycles:
    # We only care about loops that actually include the "System"
    if "System" in cycle:
        score = 0
        path_string = ""

        # Calculate Score by summing edge weights
        for i in range(len(cycle)):
            u = cycle[i]
            v = cycle[(i + 1) % len(cycle)] # Wrap around to start
            weight = G[u][v]['weight']
            score += weight
            path_string += f"{u} --({weight})--> "

        path_string += cycle[0] # Close the loop visually

        loop_data.append({
            "Loop Path": path_string,
            "Nodes": cycle,
            "Length": len(cycle),
            "Total Score": score
        })

# 6. Convert to DataFrame for Ranking
df = pd.DataFrame(loop_data)
df_sorted = df.sort_values(by="Total Score", ascending=False).reset_index(drop=True)

# 7. Display the Top 10 Loops
print("--- TOP 10 VALUE LOOPS IDENTIFIED ---")
for index, row in df_sorted.head(10).iterrows():
    print(f"Rank {index+1}: Score {row['Total Score']}")
    print(f"Path: {row['Loop Path']}")
    print("-" * 50)

--- TOP 10 VALUE LOOPS IDENTIFIED ---
Rank 1: Score 31
Path: Owners --(9)--> Pax --(9)--> System --(4)--> Insurers --(9)--> Owners
--------------------------------------------------
Rank 2: Score 29
Path: Owners --(9)--> Pax --(9)--> System --(2)--> Airports --(9)--> Owners
--------------------------------------------------
Rank 3: Score 28
Path: Owners --(6)--> Insurers --(4)--> System --(9)--> Pax --(9)--> Owners
--------------------------------------------------
Rank 4: Score 27
Path: Mfg --(9)--> System --(9)--> Investors --(9)--> Mfg
--------------------------------------------------
Rank 5: Score 27
Path: Owners --(9)--> System --(9)--> Pax --(9)--> Owners
--------------------------------------------------
Rank 6: Score 27
Path: Owners --(9)--> Pax --(9)--> System --(9)--> Owners
--------------------------------------------------
Rank 7: Score 24
Path: Schools --(9)--> System --(6)--> MRO --(9)--> Schools
--------------------------------------------------
Rank 8: Score 24
Path: O

In [ ]:
# --- PART F: STAKEHOLDER IMPORTANCE ANALYSIS (UPDATED) ---

# Method: Weighted Stakeholder Occurrence
# We calculate the "Weighted Degree" of each node in the updated graph G.
# This sums the weights (Importance * Benefit) of all Incoming and Outgoing edges.
# Note: This will now account for the split between "Owners" and "Pilots" and the new "Pax" stakeholder.

stakeholder_importance = []

# Ensure G exists (from Part E code cell)
if 'G' not in locals():
    print("Error: Graph G not found. Please run the Part E code cell first.")
else:
    for node in G.nodes():
        if node == "System":
            continue # Skip the System itself (it will obviously be #1)

        # Calculate Weighted Degree (Sum of all edge weights connected to this node)
        w_degree = G.degree(node, weight='weight')

        # Calculate simple degree (Count of connections) for comparison
        count_degree = G.degree(node)

        stakeholder_importance.append({
            "Stakeholder": node,
            "Weighted Score": w_degree,
            "Connection Count": count_degree
        })

    # Convert to DataFrame and Sort
    df_imp = pd.DataFrame(stakeholder_importance)
    df_imp_sorted = df_imp.sort_values(by="Weighted Score", ascending=False).reset_index(drop=True)

    print("--- MOST IMPORTANT STAKEHOLDERS (Weighted Occurrence) ---")
    print(df_imp_sorted)

--- MOST IMPORTANT STAKEHOLDERS (Weighted Occurrence) ---
   Stakeholder  Weighted Score  Connection Count
0       Owners              62                 8
1          Pax              36                 4
2      Schools              33                 4
3    Investors              33                 4
4          Mfg              30                 4
5          MRO              25                 4
6     Insurers              23                 4
7     Airports              17                 4
8          FAA              15                 2
9       Pilots              13                 2
10        Govt              10                 2
11        Comm               3                 2
